[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/langgraph/langgraph_orchestrator.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# LangGraph 智能体：编排器与工作智能体模式

本教程使用 LangGraph 的**编排器与工作智能体模式**构建多智能体系统，动态拆分任务，将子任务分配给专业 LLM 智能体，再综合其回复。

该模式尤其适合事先无法确定子任务结构的场景，例如模块化代码编写、多章节报告生成或研究。**编排器**负责规划与分配，**工作智能体**分别完成所分配的章节。

我们还使用 **Phoenix** 追踪和调试编排过程，直观查看编排器生成的任务、各工作智能体处理章节的方式，以及最终输出的组装过程。

完成本 Notebook 后，你将学会：
- 使用结构化输出动态规划子任务。
- 通过 LangGraph `Send` API 将子任务分配给 LLM 工作智能体。
- 收集并综合多步骤 LLM 输出。
- 使用 Phoenix 追踪并可视化编排过程。


In [ ]:
!pip install langgraph langchain langchain_community "arize-phoenix==9.0.1" arize-phoenix-otel openinference-instrumentation-langchain

In [ ]:
!pip install langchain_openai

In [ ]:
import os
from getpass import getpass

from langgraph.graph import END, START, StateGraph

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

# 配置 Phoenix 追踪
请前往 https://app.phoenix.arize.com/ 生成 API 密钥，以便使用 Phoenix 追踪 LangGraph 应用。

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="Orchestrator", auto_instrument=True)

编排器与工作智能体：研究论文生成器
----------------------------------------------
编排器规划论文的*小节*（摘要、背景等），
为每个小节创建工作智能体，最后拼接为完整草稿。

In [ ]:
import operator
from typing import Annotated, List, TypedDict

from IPython.display import Markdown
from langchain_core.messages import HumanMessage, SystemMessage

# 第 1 步：定义规划模式
首先通过 Pydantic 定义结构化输出模式，确保 LLM 在规划研究论文结构时返回格式正确、可预测的输出。

创建两个模型：

Subsection：表示论文中的一个单元，包含名称和对内容范围的简要描述。

Subsections：封装这些单元的列表。

将这些模型与 LangGraph 的 with_structured_output 配合使用，可以让编排 LLM 返回有组织的计划，供下游工作 LLM 节点可靠使用。

该模式是后续工作流的蓝图。

In [ ]:
from langgraph.types import Send
from pydantic import BaseModel, Field


class Subsection(BaseModel):
    name: str = Field(description="研究论文中此小节的名称。")
    description: str = Field(
        description="简要描述此小节将涵盖的总体主题。"
    )


class Subsections(BaseModel):
    Subsections: List[Subsection] = Field(description="研究论文的所有小节。")

# 第 2 步：设置 LLM 与工具
初始化 gpt-3.5-turbo 作为基础 LLM，并绑定 Subsections 模式以创建编排器。还加载 DuckDuckGo 搜索工具，让工作智能体利用实时网络数据充实章节。

In [ ]:
TAVILY_API_KEY = getpass("Tavily API 密钥：")
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
orchestrator_llm = llm.with_structured_output(Subsections)

search = TavilySearchResults(k=5)

# 第 3 步：定义图状态
定义两种状态模式：

State 保存整个研究论文工作流的主题、计划小节、已完成文本和最终输出。

WorkerState 保存分配给各工作智能体的单个小节任务，以及其内容的累积位置。

这一共享状态结构让 LangGraph 能协调编排器和工作智能体之间的工作。

In [ ]:
class State(TypedDict):
    topic: str
    subsections: List[Subsection]
    completed_subsections: Annotated[List[str], operator.add]
    final_paper: str
    search_results: str


class WorkerState(TypedDict):
    subsection: Annotated[Subsection, lambda x, y: y]
    completed_subsections: Annotated[List[str], operator.add]
    search_results: str

# 第 4 步：定义节点
图中定义三个核心节点：

orchestrator：通过结构化输出生成小节列表，动态规划论文结构。

subsection_writer：作为工作智能体，根据描述和范围，使用学术 Markdown 撰写一个完整小节。

synthesiser：将完成的小节合并为连贯草稿，并用分隔线隔开各节。

各节点共同构成模块化、可扩展的论文写作流水线。Phoenix 追踪允许详细检查每一步生成。

In [ ]:
def orchestrator(state: State):
    """动态规划研究论文的小节。"""
    plan = orchestrator_llm.invoke(
        [
            SystemMessage(content="为研究论文生成详细的小节计划。"),
            HumanMessage(content=f"论文主题： {state['topic']}"),
        ]
    )
    return {"subsections": plan.Subsections}


def subsection_writer(state: WorkerState):
    sub = state["subsection"]
    search_info = state.get("search_results", "")

    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "你正在撰写研究论文的小节，请将以下网络搜索结果与自身知识共同作为背景。"
                )
            ),
            HumanMessage(
                content=(
                    f"小节： {sub.name}\n"
                    f"描述： {sub.description}\n"
                    f"共享搜索结果：\n{search_info}\n\n"
                    "现在撰写该节。"
                )
            ),
        ]
    )
    return {"completed_subsections": [response.content]}


def synthesiser(state: State):
    """拼接全部已完成小节，形成最终论文草稿。"""
    full_paper = "\n\n---\n\n".join(state["completed_subsections"])
    return {"final_paper": full_paper}


def search_tool(state: State):
    query = f"{state['topic']} 研究摘要"
    search_results = search.invoke(query)
    return {"search_results": search_results}

# 第 5 步：动态分配工作智能体
此函数通过 LangGraph Send API，为每个计划小节启动独立的 subsection_writer 工作智能体。每节动态创建一个工作智能体，使系统可以根据主题复杂度灵活扩展。

这适用于事先未知章节数量的论文生成任务；Phoenix 可以独立追踪每个工作节点的输出。

In [ ]:
def assign_workers(state: State):
    """为每个计划小节启动一个 subsection_writer（共享搜索之后）。"""
    return [
        Send("subsection_writer", {"subsection": s, "search_results": state["search_results"]})
        for s in state["subsections"]
    ]

# 第 6 步：构建 LangGraph 工作流
使用 StateGraph 构建完整流水线。从 orchestrator 节点规划小节开始，经 assign_workers 动态路由到 subsection_writer 节点，最后在 synthesiser 节点汇总所有输出。

LangGraph 的条件边和 Send API 支持可扩展的并行执行。启用 Phoenix 追踪后，可以查看每个章节如何创建、跟踪并拼接。

In [ ]:
builder = StateGraph(State)

builder.add_node("orchestrator", orchestrator)
builder.add_node("search_tool", search_tool)
builder.add_node("subsection_writer", subsection_writer)
builder.add_node("synthesiser", synthesiser)

builder.add_edge(START, "orchestrator")
builder.add_edge("orchestrator", "search_tool")
builder.add_conditional_edges("search_tool", assign_workers, ["subsection_writer"])
builder.add_edge("subsection_writer", "synthesiser")
builder.add_edge("synthesiser", END)


research_paper_workflow = builder.compile()

# 第 7 步：运行研究论文生成器
使用示例主题“Scaling Laws for Large Language Models”调用编译后的 LangGraph。编排器规划提纲，各工作智能体并行撰写小节，汇总节点组装整篇论文。

集成 Phoenix 后，从章节规划到写作与汇总的每一步都有追踪，完整展示执行流程，便于调试和改进输出。

In [ ]:
research_topics = [
    "How do scaling laws impact the performance of large language models?",
    "What are the key challenges in training very large transformer models?",
    "How much data is needed to train a performant LLM?",
    "Explain the relationship between model size and accuracy in language models.",
    "Why are modern language models undertrained, and how can we fix it?",
    "What is compute-optimal training for LLMs?",
    "Compare different scaling strategies for training foundation models.",
    "How do researchers determine the best size for a transformer model?",
    "What are the trade-offs between training time and model performance?",
    "Summarize recent findings on training efficiency for large-scale language models.",
]

for topic in research_topics:
    state = research_paper_workflow.invoke({"topic": topic})

print("===== 研究论文草稿 =====\n")
Markdown(state["final_paper"])

# 第 8 步：在 Phoenix 中查看追踪！

# 添加评估（Evals）

本节评估智能体路径收敛性（Agent Path Convergence）。

**平均值（该查询所需最少步骤数 / 本次运行步骤数）**

这有助于计算编排器处理相似查询时的一致性。

参见 https://arize.com/docs/phoenix/evaluation/how-to-evals/running-pre-tested-evals/agent-path-convergence

In [ ]:
from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery

client = Client()
df = client.spans.get_spans_dataframe(
    query=SpanQuery().where("name == 'LangGraph'"), project_identifier="Orchestrator"
)
df

In [ ]:
optimal_path_length = 7  # 根据实际使用场景调整

## 生成分数

In [ ]:
import json

all_steps = []
for row in df["attributes.output.value"]:
    data = json.loads(row)
    num_subsections = len(data.get("subsections", []))
    all_steps.append(num_subsections)

convergences = []
optimal = min(all_steps)

ratios = [optimal / step for step in all_steps]

df["score"] = ratios
df["explanation"] = ["最短路径长度 / 本次路径长度"] * len(df)

In [ ]:
df

# 在 Phoenix 中查看评估

追踪顶部的 "Agent Path Convergence" 下会显示一个分数，这是已计算分数的平均值，应作为本次评估的最终指标。

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()
await px_client.spans.log_span_annotations_dataframe(
    dataframe=df,
    annotation_name="Agent Path Convergence",
    annotator_kind="LLM",
)